## Import

In [9]:
import os
import sys

import warnings
from tqdm import tqdm
import pickle

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import torchvision.transforms as T
sys.path.append('../input/pythonbox')
from box import Box
sys.path.append('../input/timm-pytorch-image-models/pytorch-image-models-master')
from timm import create_model
from torchvision.io import read_image
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from fastai.vision.all import *

import pytorch_lightning as pl
# from pytorch_lightning.utilities.seed import seed_everything
from pytorch_lightning import callbacks
# from pytorch_lightning.callbacks.progress import ProgressBarBase
from pytorch_lightning import LightningDataModule, LightningModule

sys.path.append('../input/poolformer-master')
import models as PoolFormerModels #poolformer

print(pl.__version__)
warnings.filterwarnings("ignore")

## Config

In [10]:
config_ensemble = {'exp_name':'exp073',
            'seed': 2021,
              'n_splits': 5,
              'oof_expname':{
                  1:'exp049',
                  2:'exp054',
                  3:'exp060',
                  4:'exp065',
                  5:'exp069',
                  6:'exp074',
              },
              'model':'Ridge'
}

config_ensemble = Box(config_ensemble)


#049
config_exp049 = {'exp_name':'exp049',
            'seed': 2021,
          'root': '../input/petfinder-pawpularity-score/', 
          'n_splits': 5,
          'epoch': 20,
          'transform':{
              'name': 'get_default_transforms',
              'image_size': 384
          },
          'test_loader': {
              'batch_size': 32,
              'shuffle': False,
              'num_workers': os.cpu_count(),
              'pin_memory': False,
              'drop_last': False
          },
          'model':{
              'name': 'swin_large_patch4_window12_384',
              'output_dim': 1
          },
          'loss': 'nn.BCEWithLogitsLoss',
}

config_exp049 = Box(config_exp049)

## Dataset

In [11]:
class PetfinderDataset_exp049(Dataset):
    def __init__(self, df, image_size=224):
        self._X = df["Id"].values
        self._y = None
        if "Pawpularity" in df.keys():
            self._y = df["Pawpularity"].values
        self._transform = T.Compose([
                                        T.Resize(image_size),  # 1
                                        T.CenterCrop([image_size, image_size]),  # 2
                                    ]
                                    )

    def __len__(self):
        return len(self._X)

    def __getitem__(self, idx):
        image_path = self._X[idx]
        image = read_image(image_path)
        image = self._transform(image)
        if self._y is not None:
            label = self._y[idx]
            return image, label
        return image

    
class PetfinderDataModule(LightningDataModule):
    def __init__(
        self,
        test_df,
        cfg,
    ):
        super().__init__()
        self._test_df = test_df
        self._cfg = cfg

    def __create_dataset(self, train=True):
        
        if self._cfg.exp_name=='exp049':
            return (
                PetfinderDataset_exp049(self._test_df, self._cfg.transform.image_size)
                if train
                else PetfinderDataset_exp049(self._test_df, self._cfg.transform.image_size)
            )
    def val_dataloader(self):
        dataset = self.__create_dataset(True)
        return DataLoader(dataset, **self._cfg.test_loader)
    
    def test_dataloader(self):
        dataset = self.__create_dataset(True)
        return DataLoader(dataset, **self._cfg.test_loader)

## Data Preprocess

In [12]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]  # RGB
IMAGENET_STD = [0.229, 0.224, 0.225]  # RGB


def get_default_transforms():
    transform = {
        "train": T.Compose(
            [
                T.RandomHorizontalFlip(),
                T.RandomVerticalFlip(),
                T.RandomAffine(15, translate=(0.1, 0.1), scale=(0.9, 1.1)),
                T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
                T.ConvertImageDtype(torch.float),
                T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ]
        ),
        "val": T.Compose(
            [
                T.ConvertImageDtype(torch.float),
                T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ]
        ),
    }
    return transform

def mixup(x: torch.Tensor, y: torch.Tensor, alpha: float = 1.0):
    assert alpha > 0, "alpha should be larger than 0"
    assert x.size(0) > 1, "Mixup cannot be applied to a single instance."

    lam = np.random.beta(alpha, alpha)
    rand_index = torch.randperm(x.size()[0])
    mixed_x = lam * x + (1 - lam) * x[rand_index, :]
    target_a, target_b = y, y[rand_index]
    return mixed_x, target_a, target_b, lam

## Model

In [13]:
class Model_exp049(pl.LightningModule):
    def __init__(self, cfg, val_losses=None):
        super().__init__()
        self.cfg = cfg
        self.__build_model()
        self._criterion = eval(self.cfg.loss)()
        self.transform = get_default_transforms()
        self.save_hyperparameters(cfg)
        self.val_losses = val_losses 

    def __build_model(self):
        self.backbone = create_model(
            self.cfg.model.name, pretrained=False, num_classes=0, in_chans=3
        )
        num_features = self.backbone.num_features
        self.fc = nn.Sequential(
            nn.Dropout(0.5), nn.Linear(num_features, self.cfg.model.output_dim)
        )

    def forward(self, x):
        f = self.backbone(x)
        out = self.fc(f)
        return out

    def training_step(self, batch, batch_idx):
        loss, pred, labels = self.__share_step(batch, 'train')
        self.log("train/loss", loss)
        return {'loss': loss, 'pred': pred, 'labels': labels}
        
    def validation_step(self, batch, batch_idx):
        loss, pred, labels = self.__share_step(batch, 'val')
        self.log("val/loss", loss)
        return {'pred': pred, 'labels': labels}
    
    def __share_step(self, batch, mode):
        images, labels = batch
        labels = labels.float() / 100.0
        images = self.transform[mode](images)
        
        if torch.rand(1)[0] < 0.5 and mode == 'train':
            mix_images, target_a, target_b, lam = mixup(images, labels, alpha=0.5)
            logits = self.forward(mix_images).squeeze(1)
            loss = self._criterion(logits, target_a) * lam +                 (1 - lam) * self._criterion(logits, target_b)
        else:
            logits = self.forward(images).squeeze(1)
            loss = self._criterion(logits, labels)
        
        pred = logits.sigmoid().detach().cpu() * 100.
        labels = labels.detach().cpu() * 100.
        
        return loss, pred, labels

    
def transform_get_image_shape_describe(input_df): 
    pathes = input_df['Id'].values
    shapes = []
    for path in tqdm(pathes):
        img = Image.open(path)
        height, width = img.height, img.width
        shapes.append([height, width])
    input_df[["height", "width"]] = shapes
    input_df["aspect"] = input_df["height"] / input_df["width"]
    height_mean = 904.2843018563358
    height_std = 156.90598049629264
    width_mean = 804.4262510088781
    width_std = 270.21192072081044
    if len(input_df)==8:#testだと全部同じ大きさで標準化ミスるからエラー除去でいれる。
        input_df['height'] = input_df['height'].fillna(0)
        input_df['width'] = input_df['width'].fillna(0)
    else:
        input_df['height'] = input_df['height'].apply(lambda x: (x-height_mean)/ height_std)
        input_df['width'] = input_df['width'].apply(lambda x: (x-width_mean)/ width_std)

    print(f'height_mean{height_mean}_std{height_std}')
    print(f'width_mean{width_mean}_std{width_std}')
    return input_df


def get_predict(test_loader, model, device):
    model = model.to(device)
    model = model.eval()
    predicts = []
    for images in test_loader:
        images = images.to(device)
        images = get_default_transforms()['val'](images)
        batch_size = images.size(0)
        with torch.no_grad():
            predict = model(images).sigmoid().detach().cpu().numpy() * 100
        predicts.append(predict)
    predicts = np.concatenate(predicts)
    return predicts



## Train

In [ ]:
train_df = pd.read_csv(os.path.join(config_exp049.root, "train.csv"))
test = train_df.copy()
train_df["Id"] = train_df["Id"].apply(lambda x: os.path.join(config_exp049.root, "train", x + ".jpg"))
train_df = transform_get_image_shape_describe(train_df)


In [ ]:
dataloader_exp049 = PetfinderDataModule(train_df, config_exp049).test_dataloader()


## Test

In [17]:
test_df = pd.read_csv(os.path.join(config_exp049.root, "test.csv"))
test = test_df.copy()
test_df["Id"] = test_df["Id"].apply(lambda x: os.path.join(config_exp049.root, "test", x + ".jpg"))
test_df = transform_get_image_shape_describe(test_df)
dataloader_exp049 = PetfinderDataModule(test_df, config_exp049).test_dataloader()

100%|██████████| 8/8 [00:00<00:00, 10280.16it/s]

height_mean904.2843018563358_std156.90598049629264
width_mean804.4262510088781_std270.21192072081044


In [18]:
IMG_PREDICTS = []
tmp = []
for fold in tqdm(range(config_exp049.n_splits)):
    model = Model_exp049(config_exp049)
    model_path = "../input/petfinder-334model/exp049"
    model.load_state_dict(torch.load(f'{model_path}/fold{fold}/best_loss_fold{fold}.pth'))
    predicts = get_predict(dataloader_exp049, model, 'cuda')
    tmp.append(predicts)
    
    del model
    torch.cuda.empty_cache()
IMG_PREDICTS.append(tmp)

  0%|          | 0/5 [00:00<?, ?it/s]/home/rxchi1d/anaconda3/envs/ML2/lib/python3.9/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future default), antialias=None (current default, which means False for Tensors and True for PIL), or antialias=False (only works on Tensors - PIL will still use antialiasing). This also applies if you are using the inference transforms from the models weights: update the call to weights.transforms(antialias=True).
  warnings.warn(
 20%|██        | 1/5 [00:06<00:24,  6.02s/it]/home/rxchi1d/anaconda3/envs/ML2/lib/python3.9/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing 

In [ ]:

def load_ensemble_model(path):
    with open(path, mode='rb') as fp:
        clf = pickle.load(fp)
    return clf

ensemble_predict = []
ensemble_model_exp_path = '../input/d/ktakita/petfinder-ensumble/exp085_ensumble'
for fold in range(5):
    X = np.concatenate([IMG_PREDICTS[0][fold],
                        IMG_PREDICTS[1][fold],
                        IMG_PREDICTS[2][fold],
                        IMG_PREDICTS[3][fold],
                        IMG_PREDICTS[4][fold],
                        IMG_PREDICTS[5][fold]],  
                       axis=1)
    ensemble_model_path = f'{ensemble_model_exp_path}/fold{fold}_{config_ensemble.model}.pickle'
    model = load_ensemble_model(ensemble_model_path)
    pred = model.predict(X)
    ensemble_predict.append(pred)
    
pd.DataFrame(ensemble_predict)

predict = np.mean(ensemble_predict, 0)

predictions = predict
test['Pawpularity'] = predictions
test[['Id', 'Pawpularity']].to_csv('submission_at.csv', index=False)
#display(test[['Id', 'Pawpularity']].head(8))